# ORBIT memory system full pipeline showcase

This notebook demonstrates the current ORBIT memory-system pipeline end to end using a dummy transcript.

The goal is not just to show one output, but to walk through each major memory-system capability as a separate showcase segment.


## What this notebook covers

This notebook has one dedicated section for each major capability:
1. create a deterministic memory showcase bundle
2. define a dummy transcript with multiple memory-worthy patterns
3. materialize transcript truth as ORBIT messages
4. capture turn memory step by step
5. inspect captured records and scope layering
6. inspect derived embeddings
7. run retrieval probes for multiple query intents
8. compare retrieval weighting variants
9. compare application vs postgres retrieval posture
10. inspect memory probe artifacts
11. render top-level memory status summaries
12. summarize current lifecycle and module posture


In [1]:
import sys
from pathlib import Path

ROOT = Path('/Volumes/2TB/MAS/openclaw-core/ORBIT').resolve()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd

from orbit.models import ConversationMessage, MessageRole
from orbit.notebook import (
    build_durable_bias_service,
    create_memory_showcase_bundle,
    memory_compare_backends_dataframe,
    memory_context_artifacts_dataframe,
    memory_embeddings_dataframe,
    memory_probe_dataframe,
    memory_records_dataframe,
    memory_scope_summary_dataframe,
    memory_showcase_summary_frames,
    memory_status_summary_frame,
    session_messages_dataframe,
)


## 1. Create a deterministic memory showcase bundle

This gives us a fresh store, a deterministic embedding service, and a fresh session for the pipeline walkthrough.

In [2]:
bundle = create_memory_showcase_bundle()
store = bundle['store']
service = bundle['service']
session = bundle['session']
bundle['db_path']

PosixPath('/Volumes/2TB/MAS/openclaw-core/ORBIT/.tmp/notebooks/memory_showcase/orbit.db')

## 2. Define a richer dummy transcript

This transcript intentionally mixes preference, todo, decision, and lesson-like content so the capture pipeline has multiple kinds of memory-worthy signals to work with.

In [3]:
dummy_turns = [
    {
        'user': 'I prefer concise ORBIT design answers and I usually want architecture-first explanations.',
        'assistant': 'Understood. I will keep ORBIT explanations concise and architecture-first.',
    },
    {
        'user': 'Remember to ship the transcript store before adding more fancy memory UI.',
        'assistant': 'Decision: transcript truth should stay separate from memory truth.',
    },
    {
        'user': 'I also like notebook views that make the system inspectable instead of magical.',
        'assistant': 'Lesson: retrieval results should stay visible as auxiliary context and probe artifacts.',
    },
    {
        'user': 'For long-term work, durable memory should rank above short-lived session chatter when the query is about stable preferences.',
        'assistant': 'Decision: durable memory can receive an explicit retrieval boost when the use case calls for it.',
    },
]
pd.DataFrame(dummy_turns)

,user,assistant
0,I prefer concise ORBIT design answers and I us...,Understood. I will keep ORBIT explanations con...
1,Remember to ship the transcript store before a...,Decision: transcript truth should stay separat...
2,I also like notebook views that make the syste...,Lesson: retrieval results should stay visible ...
3,"For long-term work, durable memory should rank...",Decision: durable memory can receive an explic...


## 3. Materialize transcript truth into ORBIT messages

The memory system should not replace transcript truth, so we first create normal conversation messages and store them as the canonical visible transcript.

In [4]:
message_pairs = []
turn_index = 1
for item in dummy_turns:
    user_message = ConversationMessage(
        session_id=session.session_id,
        role=MessageRole.USER,
        content=item['user'],
        turn_index=turn_index,
    )
    assistant_message = ConversationMessage(
        session_id=session.session_id,
        role=MessageRole.ASSISTANT,
        content=item['assistant'],
        turn_index=turn_index + 1,
    )
    store.save_message(user_message)
    store.save_message(assistant_message)
    message_pairs.append((user_message, assistant_message))
    turn_index += 2

session_messages_dataframe(store, session.session_id)

,message_id,session_id,turn_index,role,content,created_at,provider_message_id,metadata
0,msg_13efa0112403,session_memory_showcase,1,user,I prefer concise ORBIT design answers and I us...,2026-04-07 13:02:46.484598+00:00,None,{}
1,msg_169d6d64a266,session_memory_showcase,2,assistant,Understood. I will keep ORBIT explanations con...,2026-04-07 13:02:46.485040+00:00,None,{}
2,msg_4e7998651095,session_memory_showcase,3,user,Remember to ship the transcript store before a...,2026-04-07 13:02:46.493974+00:00,None,{}
3,msg_5801b36f7057,session_memory_showcase,4,assistant,Decision: transcript truth should stay separat...,2026-04-07 13:02:46.494002+00:00,None,{}
4,msg_3a722f3cb120,session_memory_showcase,5,user,I also like notebook views that make the syste...,2026-04-07 13:02:46.494594+00:00,None,{}
5,msg_e115651c5e98,session_memory_showcase,6,assistant,Lesson: retrieval results should stay visible ...,2026-04-07 13:02:46.494604+00:00,None,{}
6,msg_f531dc7829d5,session_memory_showcase,7,user,"For long-term work, durable memory should rank...",2026-04-07 13:02:46.495318+00:00,None,{}
7,msg_8cca33197315,session_memory_showcase,8,assistant,Decision: durable memory can receive an explic...,2026-04-07 13:02:46.495333+00:00,None,{}


## 4. Showcase turn-by-turn memory capture

Instead of bulk processing all at once, this section captures memory turn by turn so we can see what each transcript segment contributes.

In [5]:
capture_rows = []
all_captured = []
for idx, (user_message, assistant_message) in enumerate(message_pairs, start=1):
    captured = service.capture_turn_memory(
        session_id=session.session_id,
        run_id=session.conversation_id,
        user_message=user_message,
        assistant_message=assistant_message,
    )
    all_captured.extend(captured)
    capture_rows.append({
        'capture_step': idx,
        'user_excerpt': user_message.content[:80],
        'assistant_excerpt': assistant_message.content[:80],
        'captured_count': len(captured),
        'captured_scopes': [record.scope for record in captured],
        'captured_types': [record.memory_type for record in captured],
    })

pd.DataFrame(capture_rows)

,capture_step,user_excerpt,assistant_excerpt,captured_count,captured_scopes,captured_types
0,1,I prefer concise ORBIT design answers and I us...,Understood. I will keep ORBIT explanations con...,2,"[session, durable]","[summary, user_preference]"
1,2,Remember to ship the transcript store before a...,Decision: transcript truth should stay separat...,3,"[session, durable, durable]","[summary, todo, decision]"
2,3,I also like notebook views that make the syste...,Lesson: retrieval results should stay visible ...,2,"[session, durable]","[summary, lesson]"
3,4,"For long-term work, durable memory should rank...",Decision: durable memory can receive an explic...,2,"[session, durable]","[summary, decision]"


## 5. Showcase raw captured records

This is the canonical memory-record layer after capture.

In [6]:
memory_records_dataframe(store, session_id=session.session_id, limit=100)

,memory_id,scope,memory_type,source_kind,session_id,run_id,source_message_id,summary_text,detail_text,tags,salience,confidence,created_at,updated_at,archived_at,metadata
0,memory_5573724b7f0c,durable,decision,derived_summary,session_memory_showcase,run_memory_showcase,msg_8cca33197315,durable memory can receive an explicit retriev...,Decision: durable memory can receive an explic...,"[decision, policy_v2_1, multilingual_v1]",0.90,0.80,2026-04-07 13:02:53.790606+00:00,2026-04-07 13:02:53.790607+00:00,None,"{'promotion_strategy': 'policy_v2', 'promotion..."
1,memory_287a9d207774,session,summary,derived_summary,session_memory_showcase,run_memory_showcase,msg_8cca33197315,"User: For long-term work, durable memory shoul...","User: For long-term work, durable memory shoul...","[session_turn, summary]",0.40,0.70,2026-04-07 13:02:53.789606+00:00,2026-04-07 13:02:53.789606+00:00,None,"{'user_message_id': 'msg_f531dc7829d5', 'assis..."
2,memory_bd7a172084a1,durable,lesson,derived_summary,session_memory_showcase,run_memory_showcase,msg_e115651c5e98,retrieval results should stay visible as auxil...,Lesson: retrieval results should stay visible ...,"[lesson, policy_v2_1, multilingual_v1]",0.78,0.74,2026-04-07 13:02:53.789219+00:00,2026-04-07 13:02:53.789220+00:00,None,"{'promotion_strategy': 'policy_v2', 'promotion..."
3,memory_3d2a0ecbdf6c,session,summary,derived_summary,session_memory_showcase,run_memory_showcase,msg_e115651c5e98,User: I also like notebook views that make the...,User: I also like notebook views that make the...,"[session_turn, summary]",0.40,0.70,2026-04-07 13:02:53.788633+00:00,2026-04-07 13:02:53.788633+00:00,None,"{'user_message_id': 'msg_3a722f3cb120', 'assis..."
4,memory_c47849028f98,durable,decision,derived_summary,session_memory_showcase,run_memory_showcase,msg_5801b36f7057,transcript truth should stay separate from mem...,Decision: transcript truth should stay separat...,"[decision, policy_v2_1, multilingual_v1]",0.90,0.80,2026-04-07 13:02:53.787814+00:00,2026-04-07 13:02:53.787814+00:00,None,"{'promotion_strategy': 'policy_v2', 'promotion..."
5,memory_8ac35c534fbc,durable,todo,derived_summary,session_memory_showcase,run_memory_showcase,msg_5801b36f7057,ship the transcript store before adding more f...,Remember to ship the transcript store before a...,"[todo, policy_v2_1, multilingual_v1]",0.86,0.76,2026-04-07 13:02:53.787425+00:00,2026-04-07 13:02:53.787425+00:00,None,"{'promotion_strategy': 'policy_v2', 'promotion..."
6,memory_7aa2e884c84a,session,summary,derived_summary,session_memory_showcase,run_memory_showcase,msg_5801b36f7057,User: Remember to ship the transcript store be...,User: Remember to ship the transcript store be...,"[session_turn, summary]",0.40,0.70,2026-04-07 13:02:53.786908+00:00,2026-04-07 13:02:53.786909+00:00,None,"{'user_message_id': 'msg_4e7998651095', 'assis..."
7,memory_1939813aba0d,durable,user_preference,derived_summary,session_memory_showcase,run_memory_showcase,msg_169d6d64a266,I prefer concise ORBIT design answers and I us...,I prefer concise ORBIT design answers and I us...,"[user_preference, policy_v2_1, multilingual_v1]",0.82,0.80,2026-04-07 13:02:53.786422+00:00,2026-04-07 13:02:53.786424+00:00,None,"{'promotion_strategy': 'policy_v2', 'promotion..."
8,memory_8a5d5861e4c2,session,summary,derived_summary,session_memory_showcase,run_memory_showcase,msg_169d6d64a266,User: I prefer concise ORBIT design answers an...,User: I prefer concise ORBIT design answers an...,"[session_turn, summary]",0.40,0.70,2026-04-07 13:02:53.654355+00:00,2026-04-07 13:02:53.654358+00:00,None,"{'user_message_id': 'msg_13efa0112403', 'assis..."


## 6. Showcase scope/type layering

This summarizes how captured memory currently splits across session vs durable scope and across different memory types.

In [7]:
memory_scope_summary_dataframe(store, session_id=session.session_id, limit=100)

,scope,memory_type,count,avg_salience,avg_confidence
0,durable,decision,2,0.90,0.80
1,durable,lesson,1,0.78,0.74
2,durable,todo,1,0.86,0.76
3,durable,user_preference,1,0.82,0.80
4,session,summary,4,0.40,0.70


## 7. Showcase derived embedding persistence

Embeddings are not the canonical memory layer, but they are the current retrieval substrate.

In [ ]:
memory_embeddings_dataframe(store, session_id=session.session_id)

## 8. Showcase retrieval probes for different query intents

Different query phrasings should emphasize different parts of the captured memory set.

In [8]:
queries = {
    'preferences': 'what are my stable ORBIT answer preferences?',
    'todos': 'what implementation todo did I ask to remember?',
    'decisions': 'what architectural decisions were captured?',
    'inspectability': 'what did we capture about inspectable notebook memory workflows?',
}
{name: memory_probe_dataframe(store, session_id=session.session_id, query_text=query, limit=8, scope='all', memory_service=service) for name, query in queries.items()}

{'preferences':              memory_id memory_scope      memory_type  \
 0  memory_1939813aba0d      durable  user_preference   
 1  memory_8a5d5861e4c2      session          summary   
 2  memory_287a9d207774      session          summary   
 3  memory_5573724b7f0c      durable         decision   
 4  memory_c47849028f98      durable         decision   
 5  memory_8ac35c534fbc      durable             todo   
 6  memory_bd7a172084a1      durable           lesson   
 7  memory_3d2a0ecbdf6c      session          summary   
 
                                         summary_text     score  \
 0  I prefer concise ORBIT design answers and I us...  0.919571   
 1  User: I prefer concise ORBIT design answers an...  0.848571   
 2  User: For long-term work, durable memory shoul...  0.477143   
 3  durable memory can receive an explicit retriev...  0.095000   
 4  transcript truth should stay separate from mem...  0.095000   
 5  ship the transcript store before adding more f...  0.093000   
 

## 9. Showcase one retrieval probe in detail

This makes one probe easier to read than the full multi-query dict above.

In [9]:
query_text = queries['preferences']
memory_probe_dataframe(
    store,
    session_id=session.session_id,
    query_text=query_text,
    limit=8,
    scope='all',
    memory_service=service,
)

,memory_id,memory_scope,memory_type,summary_text,score,semantic_score,lexical_score,durable_boost,session_boost,salience_bonus,embedding_model,retrieval_backend,retrieval_strategy,promotion_strategy
0,memory_1939813aba0d,durable,user_preference,I prefer concise ORBIT design answers and I us...,0.919571,1.0,0.142857,0.05,0.0,0.041,demo-memory-showcase,application,hybrid_embedding_lexical_v1,policy_v2
1,memory_8a5d5861e4c2,session,summary,User: I prefer concise ORBIT design answers an...,0.848571,1.0,0.142857,0.00,0.0,0.020,demo-memory-showcase,application,hybrid_embedding_lexical_v1,None
2,memory_287a9d207774,session,summary,"User: For long-term work, durable memory shoul...",0.477143,0.5,0.285714,0.00,0.0,0.020,demo-memory-showcase,application,hybrid_embedding_lexical_v1,None
3,memory_5573724b7f0c,durable,decision,durable memory can receive an explicit retriev...,0.095000,0.0,0.000000,0.05,0.0,0.045,demo-memory-showcase,application,hybrid_embedding_lexical_v1,policy_v2
4,memory_c47849028f98,durable,decision,transcript truth should stay separate from mem...,0.095000,0.0,0.000000,0.05,0.0,0.045,demo-memory-showcase,application,hybrid_embedding_lexical_v1,policy_v2
5,memory_8ac35c534fbc,durable,todo,ship the transcript store before adding more f...,0.093000,0.0,0.000000,0.05,0.0,0.043,demo-memory-showcase,application,hybrid_embedding_lexical_v1,policy_v2
6,memory_bd7a172084a1,durable,lesson,retrieval results should stay visible as auxil...,0.089000,0.0,0.000000,0.05,0.0,0.039,demo-memory-showcase,application,hybrid_embedding_lexical_v1,policy_v2
7,memory_3d2a0ecbdf6c,session,summary,User: I also like notebook views that make the...,0.020000,0.0,0.000000,0.00,0.0,0.020,demo-memory-showcase,application,hybrid_embedding_lexical_v1,None


## 10. Showcase retrieval weighting variants

This compares the default retrieval posture against a durable-biased variant.

In [10]:
durable_biased_service = build_durable_bias_service(store=store)
default_probe = memory_probe_dataframe(
    store,
    session_id=session.session_id,
    query_text='what are my stable ORBIT answer preferences?',
    limit=8,
    scope='all',
    memory_service=service,
)
durable_probe = memory_probe_dataframe(
    store,
    session_id=session.session_id,
    query_text='what are my stable ORBIT answer preferences?',
    limit=8,
    scope='all',
    memory_service=durable_biased_service,
)
{'default': default_probe, 'durable_biased': durable_probe}

{'default':              memory_id memory_scope      memory_type  \
 0  memory_1939813aba0d      durable  user_preference   
 1  memory_8a5d5861e4c2      session          summary   
 2  memory_287a9d207774      session          summary   
 3  memory_5573724b7f0c      durable         decision   
 4  memory_c47849028f98      durable         decision   
 5  memory_8ac35c534fbc      durable             todo   
 6  memory_bd7a172084a1      durable           lesson   
 7  memory_3d2a0ecbdf6c      session          summary   
 
                                         summary_text     score  \
 0  I prefer concise ORBIT design answers and I us...  0.919571   
 1  User: I prefer concise ORBIT design answers an...  0.848571   
 2  User: For long-term work, durable memory shoul...  0.477143   
 3  durable memory can receive an explicit retriev...  0.095000   
 4  transcript truth should stay separate from mem...  0.095000   
 5  ship the transcript store before adding more f...  0.093000   
 6  r

## 11. Showcase application vs postgres backend posture

This demonstrates the current state: application retrieval executes, while postgres remains an explicit compare/stub surface.

In [11]:
memory_compare_backends_dataframe(
    store,
    session_id=session.session_id,
    query_text='what are my stable ORBIT answer preferences?',
    limit=8,
    scope='all',
    memory_service=service,
)

,memory_id,summary_text,application_score,postgres_score,delta,application_strategy,postgres_strategy
0,memory_1939813aba0d,I prefer concise ORBIT design answers and I us...,0.919571,0.0,0.919571,hybrid_embedding_lexical_v1,pgvector_reserved_stub
1,memory_8a5d5861e4c2,User: I prefer concise ORBIT design answers an...,0.848571,NaN,0.848571,hybrid_embedding_lexical_v1,None
2,memory_287a9d207774,"User: For long-term work, durable memory shoul...",0.477143,0.0,0.477143,hybrid_embedding_lexical_v1,pgvector_reserved_stub
3,memory_5573724b7f0c,durable memory can receive an explicit retriev...,0.095000,0.0,0.095000,hybrid_embedding_lexical_v1,pgvector_reserved_stub
4,memory_c47849028f98,transcript truth should stay separate from mem...,0.095000,0.0,0.095000,hybrid_embedding_lexical_v1,pgvector_reserved_stub
5,memory_8ac35c534fbc,ship the transcript store before adding more f...,0.093000,0.0,0.093000,hybrid_embedding_lexical_v1,pgvector_reserved_stub
6,memory_bd7a172084a1,retrieval results should stay visible as auxil...,0.089000,0.0,0.089000,hybrid_embedding_lexical_v1,pgvector_reserved_stub
7,memory_3d2a0ecbdf6c,User: I also like notebook views that make the...,0.020000,0.0,0.020000,hybrid_embedding_lexical_v1,pgvector_reserved_stub
8,memory_7aa2e884c84a,User: Remember to ship the transcript store be...,NaN,0.0,0.000000,None,pgvector_reserved_stub


## 12. Showcase memory probe artifacts

Probe snapshots are persisted as context artifacts so retrieval remains inspectable rather than hidden.

In [ ]:
memory_context_artifacts_dataframe(store, session.conversation_id, artifact_type='memory_probe_snapshot')

## 13. Showcase top-level summary bundle

This is the highest-level structured view over the current memory state for this transcript.

In [ ]:
frames = memory_showcase_summary_frames(
    store=store,
    service=service,
    session=session,
    query_text='what are my stable ORBIT answer preferences?',
)
frames['summary']

## 14. Showcase the rendered status card

This compresses the top-level memory posture into a dataframe-first status surface.

In [ ]:
memory_status_summary_frame(frames['summary'])

## 15. Takeaways about the current lifecycle

What this full-pipeline showcase currently demonstrates:
- transcript truth remains canonical and separate from memory truth
- memory capture can produce both session and durable records
- embeddings are derivative retrieval substrate
- retrieval is currently hybrid and inspectable
- durable-biased retrieval can be explored without changing the canonical records
- postgres retrieval is currently compare/stub posture, not the default execution path
- the notebook surface now acts like a small memory workbench, not just a one-off demo
